# markdown

> Markdown parsing utilities for splitting documents into notebook cells

In [ ]:
#| default_exp utils.md

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Tuple, Set

from nbdev_mcp.utils.re import (
    HEADER_PATTERN,
    REFERENCE_PATTERN,
    REF_LINK_USAGE_PATTERN,
    BARE_REF_PATTERN,
)

## Reference Link Extraction

Functions for handling markdown reference-style links like `[id]: url`.

In [ ]:
#| export
def extract_references(lines: List[str]) -> Tuple[List[str], Dict[str, str]]:
    """Extract reference-style link definitions from markdown lines.
    
    Reference links have the format ``[id]: url``.
    
    Parameters
    ----------
    lines : List[str]
        Lines of markdown text.
    
    Returns
    -------
    Tuple[List[str], Dict[str, str]]
        A tuple of (text_lines, ref_defs) where text_lines has reference
        definitions removed and ref_defs maps reference IDs to their
        full definition lines.
    
    Examples
    --------
    >>> lines = ['Hello [world][1]', '', '[1]: https://example.com']
    >>> text, refs = extract_references(lines)
    >>> text
    ['Hello [world][1]', '']
    >>> refs
    {'1': '[1]: https://example.com'}
    """
    refs: Dict[str, str] = {}
    text: List[str] = []
    
    for ln in lines:
        m = REFERENCE_PATTERN.match(ln)
        if m:
            refs[m.group(1)] = ln
        else:
            text.append(ln)
    
    return text, refs

In [ ]:
#| export
def refs_used(text: str, ref_defs: Dict[str, str]) -> List[str]:
    """Find which reference-style link IDs are used in the given text.
    
    Matches patterns like ``[text][id]`` or bare ``[id]`` references.
    
    Parameters
    ----------
    text : str
        Markdown text to search.
    ref_defs : Dict[str, str]
        Dictionary of known reference IDs to their definitions.
    
    Returns
    -------
    List[str]
        List of reference IDs that appear in the text.
    """
    ids: Set[str] = set()
    # Match [text][id] pattern
    for m in REF_LINK_USAGE_PATTERN.finditer(text):
        ids.add(m.group(2))
    # Match bare [id] pattern (not images, not followed by parens)
    for m in BARE_REF_PATTERN.finditer(text):
        token = m.group(1)
        if token in ref_defs:
            ids.add(token)
    return list(ids)


## Section Grouping

Functions for grouping markdown content by headings.

In [ ]:
#| export
def clear_section_buffer(
    cell_strs: List[str],
    buffer: List[str]
) -> Tuple[List[str], List[str]]:
    """Flush the buffer into cell_strs and clear it.
    
    Helper for grouping markdown content into logical sections.
    
    Parameters
    ----------
    cell_strs : List[str]
        Accumulated cell strings.
    buffer : List[str]
        Current buffer of lines to flush.
    
    Returns
    -------
    Tuple[List[str], List[str]]
        Updated (cell_strs, buffer) where buffer is cleared.
    """
    if buffer:
        cell_strs.append("\n".join(buffer).rstrip())
        buffer.clear()
    return cell_strs, buffer

In [ ]:
#| export
def group_lines_by_sections(lines: List[str]) -> List[str]:
    """Group markdown lines by heading sections.
    
    Each heading (``#``, ``##``, etc.) becomes its own section,
    and subsequent non-heading lines are grouped until the next heading.
    
    Parameters
    ----------
    lines : List[str]
        Lines of markdown text.
    
    Returns
    -------
    List[str]
        List of section strings, each representing a heading or content block.
    
    Examples
    --------
    >>> lines = ['# Title', 'Some text', '## Section', 'More text']
    >>> group_lines_by_sections(lines)
    ['# Title', 'Some text', '## Section', 'More text']
    """
    cell_strs: List[str] = []
    buffer: List[str] = []
    
    for ln in lines:
        if HEADER_PATTERN.match(ln):
            cell_strs, buffer = clear_section_buffer(cell_strs, buffer)
            cell_strs.append(ln.rstrip())
        else:
            buffer.append(ln)
    
    cell_strs, buffer = clear_section_buffer(cell_strs, buffer)
    return cell_strs

## Cell Generation

Functions for generating notebook cells from markdown.

In [ ]:
#| export
def readd_reference_links(
    cells_raw: List[str],
    ref_defs: Dict[str, str],
) -> List[Dict[str, Any]]:
    """Append reference link definitions to cells that use them.
    
    Since reference style links might be used across a large markdown cell,
    this is useful when splitting large markdown cells into sections for
    code folding.
    
    Parameters
    ----------
    cells_raw : List[str]
        Raw cell text strings.
    ref_defs : Dict[str, str]
        Reference definitions mapping IDs to full definition lines.
    
    Returns
    -------
    List[Dict[str, Any]]
        List of notebook cell dictionaries with cell_type and source.
    """
    cells: List[Dict[str, Any]] = []
    
    for cell_text in cells_raw:
        used = refs_used(cell_text, ref_defs)
        lines_out = [cell_text] if cell_text else []
        
        if used:
            if lines_out and lines_out[-1].strip():
                lines_out.append("")
            for ref in used:
                lines_out.append(ref_defs[ref])
        
        cells.append({"cell_type": "markdown", "source": "\n".join(lines_out).rstrip()})
    return cells

In [ ]:
#| export
def split_markdown_into_cells(markdown: str) -> List[Dict[str, Any]]:
    """Split a markdown document into notebook-ready markdown cells.
    
    Rules:
    
    - Each heading line (``#``, ``##``, ...) is its own cell
    - Text after a heading up to the next heading is a separate cell
    - Reference link definitions (``[id]: url``) are propagated to every
      cell that uses that id
    
    Parameters
    ----------
    markdown : str
        Full markdown document text.
    
    Returns
    -------
    List[Dict[str, Any]]
        List of notebook cell dictionaries ready to be added to a notebook.
    
    Examples
    --------
    >>> md = '''# Title
    ... 
    ... Some intro text.
    ... 
    ... ## Section 1
    ... 
    ... Content here.
    ... '''
    >>> cells = split_markdown_into_cells(md)
    >>> len(cells)
    4
    """
    lines = markdown.splitlines()
    
    # Collect reference link definitions and strip them from content
    text, refs = extract_references(lines)
    
    cell_strs = group_lines_by_sections(text)
    cells = readd_reference_links(cell_strs, refs)
    return cells

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()